In [62]:
import h5py
import numpy as np
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import torch
from torch_geometric.data import Data


def quake_h5_to_pyg(path, n_neighbors=15, n_pcs=50):
    """
    Read Quake_Smart-seq2_Trachea.h5 and convert it to a PyTorch Geometric graph.

    Parameters
    ----------
    path : str
        Path to the HDF5 file.
    n_neighbors : int
        Number of neighbors used to build the KNN graph.
    n_pcs : int
        Number of principal components used for the graph.

    Returns
    -------
    data : torch_geometric.data.Data
        PyG graph object
    adata : AnnData
        AnnData object containing the processed dataset
    """

    # --- read file ---
    with h5py.File(path, "r") as f:

        grp = f["exprs"]

        X = sp.csr_matrix(
            (grp["data"][:], grp["indices"][:], grp["indptr"][:]),
            shape=grp["shape"][:]
        )

        cells = f["obs_names"][:].astype(str)
        genes = f["var_names"][:].astype(str)

        y = f["obs"]["cluster"][:]

    # --- build AnnData ---
    adata = ad.AnnData(X)
    adata.obs_names = cells
    adata.var_names = genes
    adata.obs["cluster"] = y

    # --- Scanpy pipeline ---
    sc.pp.pca(adata, n_comps=n_pcs)
    sc.pp.neighbors(adata, n_neighbors=n_neighbors)

    # adjacency matrix
    A = adata.obsp["connectivities"]

    edge_index = np.vstack(A.nonzero())
    edge_index = torch.tensor(edge_index, dtype=torch.long)

    x = torch.tensor(adata.obsm["X_pca"], dtype=torch.float)
    y = torch.tensor(adata.obs["cluster"].values)

    data = Data(x=x, edge_index=edge_index, y=y)

    print("Cells:", data.num_nodes)
    print("Edges:", data.num_edges)
    print("Clusters:", len(np.unique(y.numpy())))

    return data, adata

In [64]:
data, adata= quake_h5_to_pyg("10X_PMBC/10X_PBMC.h5")

KeyError: "Unable to open object (object 'exprs' doesn't exist)"

In [73]:
import h5py

with h5py.File("10X_PBMC.h5", "r") as f:
    def show(name, obj):
        print(name)
    f.visititems(show)

AttributeError: 'Dataset' object has no attribute 'items'

In [ ]:
import h5py

with h5py.File("10X_PBMC.h5", "r") as f:
    def show(name, obj):
        print(name)
    f.visititems(show)

In [20]:
import h5py

def read_xy_h5(path="data/paul15.h5"):
    with h5py.File(path, "r") as f:
        X = f["X"][:]   # expression matrix
        y = f["Y"][:]   # labels

    print("X shape:", X.shape)
    print("Number of cells:", X.shape[0])

    return X, y

In [21]:
read_xy_h5()

FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = 'data/paul15.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)